# CrowdVision — Setup & Data Check

**Run this notebook first.** It will:
1. Install all required packages
2. Verify GPU / CUDA availability
3. Check every dataset is correctly placed
4. Print dataset statistics (image counts, annotation counts)
5. Show sample images with annotations

---

In [ ]:
# Auto-detect the repo root so this notebook works in JupyterLab on GCP or locally

from pathlib import Path

import sys



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

CKPT_ROOT.mkdir(exist_ok=True)

EXPS_ROOT.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))



import os

import json

import platform

import subprocess

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import torch

from PIL import Image



print(f'Repo root: {REPO_ROOT}')

print(f'Data root: {DATA_ROOT}')

In [ ]:
# ── Cell 2: Environment check ─────────────────────────────────────────────────
import torch, sys, platform

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE — using CPU"}')
print(f'OS      : {platform.system()} {platform.release()}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {DEVICE.upper()}')
if DEVICE == 'cpu':
    print('  ⚠ Training on CPU will be very slow. Use a GPU for production runs.')

In [ ]:
# ── Cell 3: Set root paths ────────────────────────────────────────────────────
from pathlib import Path

def find_repo_root(start=None):
    start_path = Path(start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start_path

REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / 'data'
CKPT_ROOT = REPO_ROOT / 'checkpoints'
CKPT_ROOT.mkdir(exist_ok=True)

import sys
sys.path.insert(0, str(REPO_ROOT))

print(f'Repo root : {REPO_ROOT}')
print(f'Data root : {DATA_ROOT}')
print(f'Checkpoints: {CKPT_ROOT}')

In [ ]:
# ── Cell 4: Dataset presence check ───────────────────────────────────────────
import os

DATASETS = {
    'ShanghaiTech-A (train)': DATA_ROOT / 'shanghaitech_part_A_final' / 'train_data' / 'images',
    'ShanghaiTech-A (test)' : DATA_ROOT / 'shanghaitech_part_A_final' / 'test_data'  / 'images',
    'ShanghaiTech-B (train)': DATA_ROOT / 'shanghaitech_part_B_final' / 'train_data' / 'images',
    'ShanghaiTech-B (test)' : DATA_ROOT / 'shanghaitech_part_B_final' / 'test_data'  / 'images',
    'JHU-Crowd++ (train)'   : DATA_ROOT / 'jhu_crowd_v2.0' / 'train' / 'images',
    'JHU-Crowd++ (val)'     : DATA_ROOT / 'jhu_crowd_v2.0' / 'val'   / 'images',
    'JHU-Crowd++ (test)'    : DATA_ROOT / 'jhu_crowd_v2.0' / 'test'  / 'images',
    'UCF-CC-50'             : DATA_ROOT / 'UCF_CC_50',
    'METR-LA'               : DATA_ROOT / 'metr-la' / 'Datasets' / 'metr-la.h5',
    'PEMS-D3'               : DATA_ROOT / 'metr-la' / 'Datasets' / 'PEMSd3.npz',
    'PEMS-D4'               : DATA_ROOT / 'metr-la' / 'Datasets' / 'pemsd4.npz',
    'PEMS-D7'               : DATA_ROOT / 'metr-la' / 'Datasets' / 'PEMSd7.npz',
    'UCSD Ped1 (Train)'     : DATA_ROOT / 'UCSD_Anomaly_Dataset.v1p2' / 'UCSDped1' / 'Train',
    'UCSD Ped2 (Train)'     : DATA_ROOT / 'UCSD_Anomaly_Dataset.v1p2' / 'UCSDped2' / 'Train',
    'Market-1501 (train)'   : DATA_ROOT / 'Market-1501-v15.09.15' / 'Market-1501-v15.09.15' / 'bounding_box_train',
}

print('Dataset Check')
print('=' * 60)
missing = []
for name, path in DATASETS.items():
    exists = path.exists()
    status = '✓' if exists else '✗ MISSING'
    n = len(list(path.glob('*'))) if exists and path.is_dir() else ('file' if exists else 0)
    print(f'  {status:10}  {name:<30}  {n}')
    if not exists:
        missing.append(name)

print()
if missing:
    print(f'⚠ {len(missing)} dataset(s) missing. Some notebooks will skip those.')
else:
    print('All datasets present ✓')

In [ ]:
# ── Cell 5: Detailed statistics per dataset ───────────────────────────────────
import scipy.io as sio
import numpy as np

def count_sha_images_and_annotations(root, split):
    img_dir = root / f'{split}_data' / 'images'
    gt_dir  = root / f'{split}_data' / 'ground_truth'
    imgs = list(img_dir.glob('IMG_*.jpg'))
    total = 0
    for img in imgs:
        gt_path = gt_dir / f'GT_{img.stem}.mat'
        if gt_path.exists():
            mat = sio.loadmat(str(gt_path))
            pts = mat['image_info'][0,0][0][0][0]
            total += len(pts)
    return len(imgs), total

print('Dataset Statistics')
print('=' * 70)

# ShanghaiTech
for part in ['A', 'B']:
    root = DATA_ROOT / f'part_{part}_final'
    if root.exists():
        for split in ['train', 'test']:
            try:
                n_imgs, n_ann = count_sha_images_and_annotations(root, split)
                print(f'  ShanghaiTech-{part} {split:5}: {n_imgs:5} images  |  {n_ann:>8,} annotations  |  avg {n_ann/max(n_imgs,1):.0f}/img')
            except Exception as e:
                print(f'  ShanghaiTech-{part} {split}: ERROR — {e}')

# JHU-Crowd++
jhu_root = DATA_ROOT / 'jhu_crowd_v2.0'
if jhu_root.exists():
    for split in ['train', 'val', 'test']:
        label_file = jhu_root / split / 'image_labels.txt'
        if label_file.exists():
            lines = label_file.read_text().strip().splitlines()
            counts = [int(l.split()[1]) for l in lines if len(l.split()) >= 2]
            print(f'  JHU-Crowd++ {split:5}: {len(counts):5} images  |  {sum(counts):>8,} annotations  |  avg {np.mean(counts):.0f}/img  max {max(counts)}')

# UCF-CC-50
ucf_root = DATA_ROOT / 'UCF_CC_50'
if ucf_root.exists():
    mats = list(ucf_root.glob('*_ann.mat'))
    counts = []
    for m in mats:
        mat = sio.loadmat(str(m))
        pts = mat.get('annPoints', np.zeros((0,2)))
        counts.append(len(pts))
    print(f'  UCF-CC-50: {len(mats):5} images  |  {sum(counts):>8,} annotations  |  avg {np.mean(counts):.0f}/img  max {max(counts)}')

# METR-LA
metr_path = DATA_ROOT / 'metr-la' / 'Datasets' / 'metr-la.h5'
if metr_path.exists():
    import h5py, pandas as pd
    with h5py.File(metr_path, 'r') as f:
        data = pd.DataFrame(f['df']['block0_values'][:])
    T, N = data.shape
    hours = T * 5 / 60
    print(f'  METR-LA:  {N} sensors  |  {T:,} timesteps (5-min)  |  {hours:.0f} hours  |  {hours/24:.1f} days')

# UCSD
for ped in ['UCSDped1', 'UCSDped2']:
    ped_root = DATA_ROOT / 'UCSD_Anomaly_Dataset.v1p2' / ped
    if ped_root.exists():
        train_seqs = len([d for d in (ped_root/'Train').iterdir() if d.is_dir() and not d.name.startswith('.')])
        test_seqs  = len([d for d in (ped_root/'Test').iterdir() if d.is_dir() and '_gt' not in d.name and not d.name.startswith('.')])
        print(f'  {ped}: {train_seqs} train sequences  |  {test_seqs} test sequences')

# Market-1501
market_root = DATA_ROOT / 'Market-1501-v15.09.15' / 'Market-1501-v15.09.15'
if market_root.exists():
    n_train = len(list((market_root/'bounding_box_train').glob('*.jpg')))
    n_test  = len(list((market_root/'bounding_box_test').glob('*.jpg')))
    n_query = len(list((market_root/'query').glob('*.jpg')))
    print(f'  Market-1501: {n_train:,} train  |  {n_test:,} gallery  |  {n_query:,} query images')

In [ ]:
# ── Cell 6: Visualise sample ShanghaiTech images ──────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy.io as sio
import numpy as np
from PIL import Image
import random

sha_root = DATA_ROOT / 'part_A_final' / 'train_data'
img_paths = sorted((sha_root / 'images').glob('IMG_*.jpg'))[:4]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('ShanghaiTech Part A — Training Samples with Head Annotations', fontsize=13)

for ax, img_path in zip(axes, img_paths):
    img = np.array(Image.open(img_path).convert('RGB'))
    gt_path = sha_root / 'ground_truth' / f'GT_{img_path.stem}.mat'
    mat = sio.loadmat(str(gt_path))
    pts = mat['image_info'][0,0][0][0][0]   # Nx2 [x, y]
    
    ax.imshow(img)
    ax.scatter(pts[:,0], pts[:,1], c='red', s=3, alpha=0.7)
    ax.set_title(f'{img_path.name}\nCount: {len(pts)}')
    ax.axis('off')

plt.tight_layout()
plt.savefig(CKPT_ROOT.parent / 'experiments' / 'sample_sha.png', dpi=100, bbox_inches='tight')
Path(CKPT_ROOT.parent / 'experiments').mkdir(exist_ok=True)
plt.show()

In [ ]:
# ── Cell 7: Visualise sample density maps ────────────────────────────────────
sys.path.insert(0, str(REPO_ROOT))
from src.data_loaders.shanghaitech import generate_density_map, load_mat_points

sha_root = DATA_ROOT / 'part_A_final' / 'train_data'
img_paths = sorted((sha_root / 'images').glob('IMG_*.jpg'))[:3]

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle('ShanghaiTech Part A — Images and Density Maps', fontsize=13)

for i, img_path in enumerate(img_paths):
    img = np.array(Image.open(img_path).convert('RGB'))
    gt_path = sha_root / 'ground_truth' / f'GT_{img_path.stem}.mat'
    pts = load_mat_points(str(gt_path))
    
    h, w = img.shape[:2]
    density = generate_density_map((h, w), pts, adaptive=True)
    
    axes[i,0].imshow(img)
    axes[i,0].set_title(f'{img_path.name}  (N={len(pts)})')
    axes[i,0].axis('off')
    
    im = axes[i,1].imshow(density, cmap='jet')
    axes[i,1].set_title(f'Density map  (sum={density.sum():.1f})')
    axes[i,1].axis('off')
    plt.colorbar(im, ax=axes[i,1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8: Verify METR-LA data ───────────────────────────────────────────────
import h5py, pandas as pd
import matplotlib.pyplot as plt

metr_path = DATA_ROOT / 'metr-la' / 'Datasets' / 'metr-la.h5'
with h5py.File(metr_path, 'r') as f:
    values = f['df']['block0_values'][:500, :5]   # first 500 timesteps, 5 sensors
    columns = f['df']['block0_items'][:5].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
for i, col in enumerate(columns):
    ax.plot(values[:, i], label=f'Sensor {col[:8]}', linewidth=0.8)
ax.set_xlabel('Timestep (5-min intervals)')
ax.set_ylabel('Speed (mph)')
ax.set_title('METR-LA — First 500 timesteps (41.7 hours), 5 sensors')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 9: Verify UCSD anomaly dataset ──────────────────────────────────────
import imageio, numpy as np, matplotlib.pyplot as plt

ucsd_root = DATA_ROOT / 'UCSD_Anomaly_Dataset.v1p2' / 'UCSDped2' / 'Train'
seq_dirs = sorted([d for d in ucsd_root.iterdir() if d.is_dir() and not d.name.startswith('.')])

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('UCSD Ped2 — Sample Training Frames', fontsize=13)

exts = {'.tif', '.tiff', '.bmp', '.png'}
for ax, seq_dir in zip(axes.flat, seq_dirs[:10]):
    frames = sorted([p for p in seq_dir.iterdir() if p.suffix.lower() in exts and not p.name.startswith('.')])
    if frames:
        img = imageio.imread(str(frames[len(frames)//2]))
        ax.imshow(img, cmap='gray')
        ax.set_title(f'{seq_dir.name}  ({len(frames)} frames)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: Import all src modules to verify installation ────────────────────
print('Verifying src imports...')

from src.data_loaders import (
    ShanghaiTechDataset, JHUCrowdDataset, UCFCC50Dataset,
    load_metr_la, get_ucsd_loaders, Market1501Dataset
)
print('  ✓ data_loaders')

from src.models import (
    CSRNet, CSRNetLite, AdaptiveCSRNet,
    GCNGRU, AdaptiveNASGNN,
    ConvAE, ConvLSTMAE, FutureFrameNet,
    UnifiedCrowdVision
)
print('  ✓ models')

from src.losses import CombinedDensityLoss
print('  ✓ losses')

from src.training import DensityTrainer, ForecastingTrainer, AnomalyTrainer
print('  ✓ training')

from src.evaluation import evaluate_density, evaluate_forecasting, evaluate_anomaly_detection
print('  ✓ evaluation')

from src.utils import show_density_map, plot_training_curves, make_results_table
print('  ✓ utils')

print('\nAll imports successful! Ready to train. ✓')

---
## Setup Complete!

### Next Steps  
Run notebooks in order:
- **01_density_estimation.ipynb** — Train CSRNet and AdaptiveCSRNet on ShanghaiTech + JHU
- **02_forecasting.ipynb** — Train GCN-GRU and AdaptiveNAS-GNN on METR-LA
- **03_anomaly_detection.ipynb** — Train ConvAE and FutureFrameNet on UCSD
- **04_tracking.ipynb** — Train re-ID model on Market-1501, run DeepSORT
- **05_multitask_training.ipynb** — Unified multi-task model
- **06_evaluation_and_paper_results.ipynb** — Generate all paper tables and figures